# Story 6: A Pipeline whose Steps Need Different Resources
Your workflow has three steps: a serial structure preparation, an MPI-parallel simulation and a serial analysis -
repeated for several lattice constants, followed by a final step which selects the best candidate. You want to give
every step exactly the resources it needs, and have each step start as soon as its inputs are ready.

**With plain SLURM** every step needs its own batch script with its own `#SBATCH` header. You capture the job ID of
each submission, pass it to the next one as `--dependency=afterok:<jobid>`, and write the intermediate results to
files only to parse them again. Running the whole workflow in one large MPI allocation avoids the bookkeeping, but
wastes core hours while the serial steps run.

**With executorlib** you define the resources per function call with the `resource_dict`. Passing the future of one
call as argument to the next one turns into a SLURM job dependency, and the result is handed over through the cache.

This notebook runs on the **login node** of a SLURM cluster.

## Cluster-specific settings
**Adapt the following cell to your cluster.** It collects everything which depends on the cluster rather than on your science. The values below are the ones used to test this notebook on a single-node SLURM test cluster. The submission template is a [Jinja2](https://jinja.palletsprojects.com) template which executorlib renders for every function call: add the lines your cluster requires, for example `#SBATCH --partition=<name>`, `#SBATCH --account=<project>` or `module load ...` commands before `{{command}}`. The [executorlib-cluster-setup](https://executorlib.readthedocs.io/en/latest/2-hpc-cluster.html#setup-with-claude-code) skill for Claude Code can analyse your cluster and generate a tested template.

The `pmi_mode` selects the process management interface which `srun` uses to start MPI-parallel functions. It depends on the MPI library your Python environment is linked to.

In [ ]:
# Cluster-specific settings - adapt these values to your cluster.
submission_template = """\
#!/bin/bash
#SBATCH --output=time.out
#SBATCH --job-name={{job_name}}
#SBATCH --chdir={{working_directory}}
#SBATCH --get-user-env
#SBATCH --ntasks={{cores}}
{%- if run_time_max %}
#SBATCH --time={{ [1, run_time_max // 60]|max }}
{%- endif %}
{%- if dependency_list %}
#SBATCH --dependency=afterok:{{ dependency_list | join(',') }}
{%- endif %}

{{command}}
"""
pmi_mode = "pmi2"  # MPI process management interface: "pmi2" for MPICH, "pmix" for OpenMPI >= 5

## Step 1: Write the steps as Python functions
Only the simulation uses MPI via [mpi4py](https://mpi4py.readthedocs.io), the other steps are serial. An MPI-parallel
function is executed on every MPI rank, so executorlib returns a list with one result per rank.

In [ ]:
def prepare_structure(lattice_constant):
    # Stand-in for building an atomistic structure (serial, seconds).
    positions = [lattice_constant * i for i in range(8)]
    return {"positions": positions, "n_steps": 100}


def run_simulation(positions, n_steps):
    # Stand-in for an MPI-parallel simulation code (hours): every rank handles a slice of the atoms.
    from mpi4py import MPI

    comm = MPI.COMM_WORLD
    local_energy = sum(-1.0 / (1.0 + x) for x in positions[comm.Get_rank()::comm.Get_size()])
    energy = comm.allreduce(local_energy) * n_steps / 100
    return {"energy": energy, "mpi_ranks": comm.Get_size()}


def analyse(simulation_result, lattice_constant):
    # Stand-in for the post-processing of the simulation output (serial).
    return {"lattice_constant": lattice_constant, **simulation_result}


def select_minimum(*analysis_results):
    # Final many-to-one step: the lattice constant with the lowest energy.
    return min(analysis_results, key=lambda result: result["energy"])

## Step 2: Define the resources per step
The serial steps get one core, the simulation gets `cores_mpi` MPI ranks. `run_time_max` (in seconds) becomes the
`#SBATCH --time` limit of the corresponding job, so short steps can benefit from backfilling.

In [ ]:
cores_mpi = 2
resources_serial = {"cores": 1, "run_time_max": 60}
resources_mpi = {"cores": cores_mpi, "run_time_max": 120}

On a production cluster the simulation might rather request:

```python
resources_mpi = {
    "cores": 256,             # MPI ranks, rendered as {{cores}} in the template
    "threads_per_core": 2,    # executorlib requests cores * threads_per_core CPUs in total
    "num_nodes": 2,           # available as {{num_nodes}} - add "#SBATCH --nodes={{num_nodes}}" to the template
    "run_time_max": 4 * 3600,
}
```

Every key of the `resource_dict` is available in the submission template, so cluster-specific options like the
partition, the memory or GPUs can be added there. Note that the `SlurmClusterExecutor` does not translate
`gpus_per_core` into the job script by itself - add a line like `#SBATCH --gpus-per-task={{gpus}}` to the template
and request `{"gpus": 1}`. Inside an allocation the [SlurmJobExecutor](story-2-parameter-study.ipynb) translates
`gpus_per_core` directly to `srun --gpus-per-task`.

## Step 3: Connect the steps with futures
The workflow is written as a function which takes the executor as argument, so it can be plotted before it is
executed. Three utilities connect the steps without waiting for any result:

* `get_item_from_future()` selects one entry of a dictionary returned by a future,
* `split_future()` splits a future of a list or tuple - here the per-rank results of the MPI function,
* futures passed as arguments become `--dependency=afterok:<jobid>` of the dependent job.

In [ ]:
from executorlib import get_item_from_future, split_future


def submit_workflow(exe, lattice_constant_lst):
    analysis_lst = []
    for lattice_constant in lattice_constant_lst:
        structure = exe.submit(prepare_structure, lattice_constant, resource_dict=resources_serial)
        simulation = exe.submit(
            run_simulation,
            positions=get_item_from_future(structure, key="positions"),
            n_steps=get_item_from_future(structure, key="n_steps"),
            resource_dict=resources_mpi,
        )
        rank_results = split_future(simulation, n=cores_mpi)  # one result per MPI rank
        analysis_lst.append(
            exe.submit(analyse, rank_results[0], lattice_constant, resource_dict=resources_serial)
        )
    return exe.submit(select_minimum, *analysis_lst, resource_dict=resources_serial)

With `plot_dependency_graph=True` nothing is submitted - the executor only records the calls and draws the graph,
which is a cheap way to check a workflow before spending compute time. This requires the optional
[pygraphviz](https://pygraphviz.github.io) package.

In [ ]:
from executorlib import SlurmClusterExecutor

cache_directory = "./story2_cache"
lattice_constant_lst = [3.9, 4.1]

with SlurmClusterExecutor(
    cache_directory=cache_directory,
    pmi_mode=pmi_mode,
    resource_dict={"submission_template": submission_template},
    plot_dependency_graph=True,
) as exe:
    submit_workflow(exe, lattice_constant_lst)

## Step 4: Run the pipeline
All seven jobs are submitted immediately. SLURM holds each job until the jobs it depends on finished successfully, so
the Python process does not have to coordinate anything - combined with [Story 1](story-1-submit-and-reconnect.ipynb)
you could even disconnect right after the submission.

In [ ]:
with SlurmClusterExecutor(
    cache_directory=cache_directory,
    pmi_mode=pmi_mode,
    resource_dict={"submission_template": submission_template},
) as exe:
    best = submit_workflow(exe, lattice_constant_lst)
    print(best.result())

## Step 5: Verify which resources each step received
The cache records the `resource_dict` and the SLURM job ID `queue_id` of every call, which can be passed to
`sacct -j <queue_id>` to inspect the accounting record of the job.

In [ ]:
import pandas
from executorlib import get_cache_data

df = pandas.DataFrame(get_cache_data(cache_directory=cache_directory))
df["function"] = df["function"].apply(lambda fn: fn.__name__)
df["cores"] = df["resource_dict"].apply(lambda resource_dict: resource_dict["cores"])
df[["function", "cores", "queue_id", "runtime"]].sort_values("queue_id")

## Clean up

In [ ]:
import shutil

shutil.rmtree(cache_directory, ignore_errors=True)

## What you learned
* The `resource_dict` passed to `submit()` assigns MPI ranks, threads, run time and any template variable per call.
* `pmi_mode` defines how MPI-parallel functions are started with `srun`.
* Futures used as arguments become SLURM job dependencies; `get_item_from_future()` and `split_future()` connect
  individual entries of a result without waiting for it.
* `plot_dependency_graph=True` draws the workflow without submitting anything.

**Reference:** [HPC Cluster Executor](https://executorlib.readthedocs.io/en/latest/2-hpc-cluster.html),
[dependencies](https://executorlib.readthedocs.io/en/latest/1-single-node.html#dependencies), [split future objects](https://executorlib.readthedocs.io/en/latest/1-single-node.html#split-future-objects),
[MPI parallel functions](https://executorlib.readthedocs.io/en/latest/1-single-node.html#mpi-parallel-functions) and the
[resource dictionary](https://executorlib.readthedocs.io/en/latest/trouble_shooting.html#resource-dictionary).